# Notebook 11 . Olah CSV Berukuran Besar

Baca dan salin CSV sedikit demi sedikit agar penggunaan memori lebih ringan. Bisa memilih beberapa kolom dan menyaring satu nilai. Proses ini tidak menyimpan seluruh file di memori sekaligus.

In [ ]:
from pathlib import Path
import pandas as pd

# PENGATURAN YANG BISA DIUBAH
Input_CSV = Path("Input/input_besar.csv")
Separator_CSV = ","  # Ganti menjadi ";" jika CSV memakai titik koma
Ukuran_Bagian = 100_000
Kolom_Diproses = None        # Contoh: ["id", "tanggal", "nilai"] atau None untuk semua kolom
Kolom_Filter = None      # Contoh: "status"; None berarti tanpa filter
Nilai_Filter = None       # Contoh: "aktif"
Output_CSV = Path("Output/hasil_besar.csv")
Timpa_File_Output = False  # Ubah ke True jika memang ingin mengganti hasil lama

def find_project_dir():
    current = Path.cwd().resolve()
    for folder in (current, *current.parents):
        if (folder / "Input").is_dir():
            (folder / "Output").mkdir(parents=True, exist_ok=True)
            return folder
    return current

BASE_DIR = find_project_dir()


## Proses per bagian

Sel ini menulis hasil baru. Jangan gunakan `Output_CSV` yang sama dengan file sumber.

In [ ]:
source = Input_CSV if Input_CSV.is_absolute() else BASE_DIR / Input_CSV
output = Output_CSV if Output_CSV.is_absolute() else BASE_DIR / Output_CSV
if not source.exists():
    raise FileNotFoundError(f"File tidak ditemukan: {source}")
if source.resolve() == output.resolve():
    raise ValueError("Pilih Output_CSV yang berbeda agar file sumber tidak tertimpa.")
if output.exists() and not Timpa_File_Output:
    raise FileExistsError(f"File hasil sudah ada: {output}. Ubah Output_CSV atau atur Timpa_File_Output menjadi True.")
if Kolom_Filter and Kolom_Diproses is not None and Kolom_Filter not in Kolom_Diproses:
    Kolom_Diproses = [*Kolom_Diproses, Kolom_Filter]
output.parent.mkdir(parents=True, exist_ok=True)
if output.exists(): output.unlink()
total_read = total_written = 0
for chunk_number, chunk in enumerate(
    pd.read_csv(
        source,
        sep=Separator_CSV,
        dtype="string",
        encoding="utf-8-sig",
        usecols=Kolom_Diproses,
        chunksize=Ukuran_Bagian,
    ),
    start=1,
):
    total_read += len(chunk)
    if Kolom_Filter:
        if Kolom_Filter not in chunk:
            raise KeyError(f"Kolom filter tidak ditemukan: {Kolom_Filter}")
        chunk = chunk.loc[chunk[Kolom_Filter].eq(Nilai_Filter)]
    chunk.to_csv(
        output,
        mode="w" if chunk_number == 1 else "a",
        header=(chunk_number == 1),
        index=False,
        encoding=("utf-8-sig" if chunk_number == 1 else "utf-8"),
        sep=Separator_CSV,
    )
    total_written += len(chunk)
    if chunk_number % 10 == 0:
        print(f"Bagian {chunk_number}: dibaca {total_read:,}, ditulis {total_written:,} baris")
if total_read == 0:
    # Buat file kosong dengan header bila CSV sumber tidak memiliki baris data.
    header = pd.read_csv(source, sep=Separator_CSV, nrows=0, encoding="utf-8-sig", usecols=Kolom_Diproses)
    header.to_csv(output, index=False, encoding="utf-8-sig", sep=Separator_CSV)
print(
    f"Selesai. Dibaca {total_read:,}, "
    f"ditulis {total_written:,} baris -> {output.resolve()}"
)
